In [ ]:
# Install core machine learning and cheminformatics libraries
!pip install torch torchvision torchaudio
!pip install torch_geometric
!pip install dgl-lifesci
!pip install deepchem
!pip install rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.4 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement dgl-lifesci (from versions: none)
ERROR: No matching distribution found for dgl-lifesci
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 25.4 MB/s eta 0:00:00


In [ ]:
!pip install chembl_webresource_client pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 909.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the Data
# train_data = pd.read_csv('/content/02_testing.csv')

train_data = pd.read_csv('/content/drive/MyDrive/Drug-food_project/00_training.csv')
# test_data = pd.read_csv('/content/00_training.csv')

test_data = pd.read_csv('/content/drive/MyDrive/Drug-food_project/01_validation.csv')

# 2. Separate the "Features" (X) from the "Answers" (y)
# Drop the LABEL column to create the features, keep it for the target
X_train = train_data.drop('LABEL', axis=1)
y_train = train_data['LABEL']

# Do the exact same for the Final Exam (Testing Data)
X_test = test_data.drop('LABEL', axis=1)
y_test = test_data['LABEL'] # <- This is your Answer Key!

# 3. Create and Train the XGBoost Model (The Studying Phase)
print("Training the model... (This might take a minute)")
model = xgb.XGBClassifier(
    n_estimators=100,      # How many decision trees to build
    learning_rate=0.1,     # How fast the model learns
    random_state=42        # Ensures we get the same result every time
)
model.fit(X_train, y_train)

# 4. The Final Exam (Testing Phase)
# We ONLY give it X_test (the features). We hide y_test (the answers).
print("Taking the Final Exam...")
predictions = model.predict(X_test)

# 5. Grade the Exam!
# We compare the AI's guesses (predictions) against the true answers (y_test)
accuracy = accuracy_score(y_test, predictions)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

# Print a detailed report card showing how well it did on 0, 1, and 2
print("\nDetailed Report Card:")
print(classification_report(y_test, predictions))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Drug-food_project/00_training.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score

train_df = pd.read_csv('/content/drive/MyDrive/00_training.csv')
val_df = pd.read_csv('/content/drive/MyDrive/01_validation.csv')
test_df = pd.read_csv('/content/drive/MyDrive/02_testing.csv')
external_df = pd.read_csv('/content/drive/MyDrive/03_external_test_set.csv')

X_train, y_train = train_df.drop('LABEL', axis=1), train_df['LABEL']
X_val, y_val = val_df.drop('LABEL', axis=1), val_df['LABEL']
X_test, y_test = test_df.drop('LABEL', axis=1), test_df['LABEL']
X_ext, y_ext = external_df.drop('LABEL', axis=1), external_df['LABEL']

print("Training the balanced model...")
model = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=0.1,
    reg_alpha=0.0,
    early_stopping_rounds=50
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

try:
    model.save_model("drug_food_model.json")
    print("✅ SUCCESS! 'drug_food_model.json' has been created and saved!")
except FileNotFoundError:
    print("❌ Error: Please make sure '01_validation.csv' is uploaded to Colab first.")

print("\n--- Model Evaluation ---")

test_predictions = model.predict(X_test)
test_acc = accuracy_score(y_test, test_predictions)
print(f"Testing Accuracy (Internal): {test_acc * 100:.2f}%")

ext_predictions = model.predict(X_ext)
ext_acc = accuracy_score(y_ext, ext_predictions)
print(f"External Accuracy (Real-World): {ext_acc * 100:.2f}%")



# Download it to your computer
from google.colab import files
files.download("xgboost_model.json")

Training the balanced model...


KeyboardInterrupt: 

In [ ]:
import requests
import pandas as pd
import xgboost as xgb
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def get_chembl_smiles(drug_name):
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=20)
            if response.status_code == 200:
                data = response.json()
                if data.get('molecules'):
                    smiles = data['molecules'][0]['molecule_structures'].get('canonical_smiles')
                    if smiles:
                        return smiles
            return None
        except requests.exceptions.Timeout:
            time.sleep(3)
        except Exception:
            return None
    return None

def get_foodb_smiles(compound_name, csv_path='/content/drive/MyDrive/Compound.csv'):
    try:
        df = pd.read_csv(csv_path, usecols=['name', 'smiles'], low_memory=False)
        match = df[df['name'].str.lower() == compound_name.lower()]
        if not match.empty:
            return match.iloc[0]['smiles']
    except Exception:
        pass
    return None

def get_base_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return None
    return {
        'MTPSA': Descriptors.TPSA(mol),
        'LabuteASA': rdMolDescriptors.CalcLabuteASA(mol),
        'MRVSA': rdMolDescriptors.SMR_VSA_(mol),
        'PEOEVSA': rdMolDescriptors.PEOE_VSA_(mol),
        'slogPVSA': rdMolDescriptors.SlogP_VSA_(mol),
        'EstateVSA': rdMolDescriptors.EState_VSA_(mol),
        'VSAEstate': rdMolDescriptors.VSA_EState_(mol)
    }

def calculate_18_features(drug_smiles, food_smiles):
    d = get_base_features(drug_smiles)
    f = get_base_features(food_smiles)
    if not d or not f:
        return None

    features = [
        d['MTPSA'] + f['MTPSA'],
        d['MRVSA'][9] + f['MRVSA'][9],
        d['MRVSA'][8] + f['MRVSA'][8],
        d['MRVSA'][0] + f['MRVSA'][0],
        d['MRVSA'][2] + f['MRVSA'][2],
        d['VSAEstate'][10] + f['VSAEstate'][10],
        d['EstateVSA'][0] * f['LabuteASA'],
        d['PEOEVSA'][12] + f['PEOEVSA'][12],
        d['PEOEVSA'][10] + f['PEOEVSA'][10],
        d['PEOEVSA'][5] + f['PEOEVSA'][5],
        d['PEOEVSA'][9] + f['PEOEVSA'][9],
        d['slogPVSA'][2] + f['slogPVSA'][2],
        d['slogPVSA'][0] + f['slogPVSA'][0],
        d['slogPVSA'][9] + f['slogPVSA'][9],
        d['VSAEstate'][7] + f['VSAEstate'][7],
        d['EstateVSA'][7] + f['EstateVSA'][7],
        d['EstateVSA'][2] + f['EstateVSA'][2],
        d['EstateVSA'][1] * f['VSAEstate'][8]
    ]
    return features

def run_prediction_engine():
    try:
        model = xgb.XGBClassifier()
        model.load_model("drug_food_model.json")
    except Exception:
        return

    try:
        feature_names = pd.read_csv('/content/drive/MyDrive/01_validation.csv', nrows=0).drop('LABEL', axis=1).columns.tolist()
    except Exception:
        return

    while True:
        drug_name = input("Enter Drug Name (or 'quit' to exit): ").strip()
        if drug_name.lower() == 'quit':
            break

        food_compound = input("Enter Food Compound: ").strip()
        if food_compound.lower() == 'quit':
            break

        drug_smiles = get_chembl_smiles(drug_name)
        if not drug_smiles: continue

        food_smiles = get_foodb_smiles(food_compound)
        if not food_smiles: continue

        features_array = calculate_18_features(drug_smiles, food_smiles)
        if not features_array: continue

        input_df = pd.DataFrame([features_array], columns=feature_names)
        prediction_label = int(model.predict(input_df)[0])

        print(f"PREDICTION LABEL: {prediction_label}")

        if prediction_label == 0:
            print("🟢 SAFE")
        elif prediction_label == 1:
            print("🟡 WARNING")
        elif prediction_label == 2:
            print("🔴 DANGER")

if __name__ == "__main__":
    run_prediction_engine()

In [ ]:
!pip install rdkit xgboost pandas requests numpy

In [ ]:
import requests
import pandas as pd
import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

print("Libraries loaded successfully!")

In [ ]:
def get_chembl_smiles(drug_name):
    """Fetches the SMILES string for a drug using the ChEMBL API."""
    print(f"[*] Querying ChEMBL API for '{drug_name}'...")
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('molecules'):
                smiles = data['molecules'][0]['molecule_structures'].get('canonical_smiles')
                if smiles:
                    return smiles
        print(f"[!] Could not find valid SMILES for drug: {drug_name}")
    except Exception as e:
        print(f"[!] ChEMBL API Error: {e}")
    return None

def get_foodb_smiles(compound_name, csv_path='/content/drive/MyDrive/Compound.csv'):
    """Looks up the SMILES string for a food compound in the local FooDB CSV."""
    print(f"[*] Searching local FooDB database for '{compound_name}'...")
    try:
        df = pd.read_csv(csv_path, usecols=['name', 'moldb_smiles'], low_memory=False)
        match = df[df['name'].str.lower() == compound_name.lower()]

        if not match.empty:
            return match.iloc[0]['moldb_smiles']
        else:
            print(f"[!] Compound '{compound_name}' not found in {csv_path}")
    except FileNotFoundError:
        print(f"[!] Error: The file '{csv_path}' was not found in the directory.")
    return None

In [ ]:
def get_base_features(smiles):
    """Calculates all necessary RDKit descriptors for a single SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    return {
        'MTPSA': Descriptors.TPSA(mol),
        'LabuteASA': rdMolDescriptors.CalcLabuteASA(mol),
        'MRVSA': rdMolDescriptors.SMR_VSA_(mol),
        'PEOEVSA': rdMolDescriptors.PEOE_VSA_(mol),
        'slogPVSA': rdMolDescriptors.SlogP_VSA_(mol),
        'EstateVSA': rdMolDescriptors.EState_VSA_(mol),
        'VSAEstate': rdMolDescriptors.VSA_EState_(mol)
    }

def calculate_18_features(drug_smiles, food_smiles):
    """Combines drug and food features mathematically to match the training data."""
    print("[*] Calculating molecular features using RDKit...")
    d = get_base_features(drug_smiles)
    f = get_base_features(food_smiles)

    if not d or not f:
        return None

    features = [
        d['MTPSA'] + f['MTPSA'],
        d['MRVSA'][9] + f['MRVSA'][9],
        d['MRVSA'][8] + f['MRVSA'][8],
        d['MRVSA'][0] + f['MRVSA'][0],
        d['MRVSA'][2] + f['MRVSA'][2],
        d['VSAEstate'][10] + f['VSAEstate'][10],
        d['EstateVSA'][0] * f['LabuteASA'],
        d['PEOEVSA'][12] + f['PEOEVSA'][12],
        d['PEOEVSA'][10] + f['PEOEVSA'][10],
        d['PEOEVSA'][5] + f['PEOEVSA'][5],
        d['PEOEVSA'][9] + f['PEOEVSA'][9],
        d['slogPVSA'][2] + f['slogPVSA'][2],
        d['slogPVSA'][0] + f['slogPVSA'][0],
        d['slogPVSA'][9] + f['slogPVSA'][9],
        d['VSAEstate'][7] + f['VSAEstate'][7],
        d['EstateVSA'][7] + f['EstateVSA'][7],
        d['EstateVSA'][2] + f['EstateVSA'][2],
        d['EstateVSA'][1] * f['VSAEstate'][8]
    ]
    return features

In [ ]:

try:
    model = xgb.XGBClassifier()
    model.load_model("drug_food_model.json")
    print("✅ Model 'drug_food_model.json' loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")

try:
    feature_names = pd.read_csv('/content/drive/MyDrive/01_validation.csv', nrows=0).drop('LABEL', axis=1).columns.tolist()
    print("✅ Feature columns loaded successfully!")
except FileNotFoundError:
    print("❌ Error: '01_validation.csv' missing. Cannot map feature names.")

def predict_interaction(drug_name, food_compound):
    print("\n" + "="*50)
    print(f"  ANALYZING: {drug_name.upper()} + {food_compound.upper()}")
    print("="*50)

    drug_smiles = get_chembl_smiles(drug_name)
    food_smiles = get_foodb_smiles(food_compound)

    if not drug_smiles or not food_smiles:
        return

    print(f"\n[+] SMILES Retrieved:")
    print(f"    Drug: {drug_smiles[:40]}...")
    print(f"    Food: {food_smiles[:40]}...")

    features_array = calculate_18_features(drug_smiles, food_smiles)
    if not features_array:
        return

    input_df = pd.DataFrame([features_array], columns=feature_names)

    print("\n[*] Running XGBoost Inference...")
    prediction_label = int(model.predict(input_df)[0])

    print("\n" + "="*40)
    if prediction_label == 0:
        print("🟢 RESULT: SAFE (No significant interaction)")
    elif prediction_label == 1:
        print("🟡 RESULT: WARNING (Minor interaction possible)")
    elif prediction_label == 2:
        print("🔴 RESULT: DANGER (Severe adverse event predicted)")
    print("="*40)

In [ ]:

DRUG = "Phenelzine"
FOOD = "Pelargonidin 3-arabinoside"

predict_interaction(DRUG, FOOD)

In [ ]:
import pandas as pd

# Let's say this is your dataframe
df = pd.read_csv('/content/drive/MyDrive/Compound.csv', low_memory=False)

# Define the columns you actually want
to_keep = ['public_id', 'name', 'cas_number','moldb_smiles']

# Overwrite the dataframe with only those columns
df = df[to_keep]

In [ ]:
output_csv_path = '/content/drive/MyDrive/Compound_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"Modified DataFrame saved to: {output_csv_path}")

# Display the first few rows of the saved DataFrame to confirm
df_saved = pd.read_csv(output_csv_path)
display(df_saved.head())

Modified DataFrame saved to: /content/drive/MyDrive/Compound_processed.csv


,public_id,name,cas_number,moldb_smiles
0,FDB000004,Cyanidin 3-(6''-acetyl-galactoside),[H][C@]1(COC(C)=O)O[C@@]([H])(OC2=CC3=C(O)C=C(...,HBXXDBKJLPLXPR-DLBZZEGUSA-O
1,FDB000013,Cyanidin 3-(6''-succinyl-glucoside),[H][C@]1(COC(=O)CCC(O)=O)O[C@@]([H])(OC2=CC3=C...,MIYGQTFETYBMKF-WVXUANQFSA-O
2,FDB000014,Pelargonidin 3-(6''-succinyl-glucoside),[H][C@]1(COC(=O)CCC(O)=O)O[C@@]([H])(OC2=CC3=C...,UBUSYXLSGMWUJJ-WVXUANQFSA-O
3,FDB000024,Petunidin 3-O-(6''-acetyl-galactoside),[H][C@]1(COC(C)=O)OC(OC2=C([O+]=C3C=C(O)C=C(O)...,GPUBWXUQPURXOQ-SKKXNPCDSA-O
4,FDB000025,Peonidin 3-(6''-acetyl-galactoside),[H][C@]1(COC(C)=O)OC(OC2=C([O+]=C3C=C(O)C=C(O)...,MBSKDCPWFSMEFD-ZKVZURMCSA-O


In [ ]:
!pip install rdkit
from rdkit import Chem

smiles = "[H][C@]1(COC(C)=O)OC(OC2=C([O+]=C3C=C(O)C=C(O)C3=C2)C2=CC(OC)=C(O)C(OC)=C2)[C@]([H])(O)[C@@]([H])(O)[C@@]1([H])O"

# Convert SMILES to a Molecule object
mol = Chem.MolFromSmiles(smiles)

# Convert that Molecule back to an InChIKey
generated_key = Chem.MolToInchiKey(mol)

print(f"Generated Key: {generated_key}")
# It should match: WGYWEDJQQFKGID-ZTKYFMRJSA-O

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 17.4 MB/s eta 0:00:00
Generated Key: WGYWEDJQQFKGID-ZTKYFMRJSA-O


In [ ]:
import pandas as pd

# Let's say this is your dataframe
df = pd.read_csv('/content/drive/MyDrive/PubChem_compound_FDA_approved_drugs.csv', low_memory=False)

# Define the columns you actually want
to_keep = ['Compound_CID', 'Name', 'SMILES','InChIKey']

# Overwrite the dataframe with only those columns
df = df[to_keep]

In [ ]:
output_csv_path = '/content/drive/MyDrive/pubchem_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"Modified DataFrame saved to: {output_csv_path}")

# Display the first few rows of the saved DataFrame to confirm
df_saved = pd.read_csv(output_csv_path)
display(df_saved.head())

Modified DataFrame saved to: /content/drive/MyDrive/pubchem_processed.csv


,Compound_CID,Name,SMILES,InChIKey
0,135398737,Clozapine,CN1CCN(CC1)C2=NC3=C(C=CC(=C3)Cl)NC4=CC=CC=C42,QZUDBNBUXVUHMW-UHFFFAOYSA-N
1,2391,Bisacodyl,CC(=O)OC1=CC=C(C=C1)C(C2=CC=C(C=C2)OC(=O)C)C3=...,KHOITXIGCFIULA-UHFFFAOYSA-N
2,2160,Amitriptyline,CN(C)CCC=C1C2=CC=CC=C2CCC3=CC=CC=C31,KRMDCWKBEZIMAB-UHFFFAOYSA-N
3,71285,"N-Methyl-3,4-methylenedioxyamphetamine Hydroch...",CC(CC1=CC2=C(C=C1)OCO2)NC.Cl,LUWHVONVCYWRMZ-UHFFFAOYSA-N
4,89594,Nicotine,CN1CCC[C@H]1C2=CN=CC=C2,SNICXCGAKADSCV-JTQLQIEISA-N


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/00_training.csv')

# 1. Add the new column (e.g., a simple flag)
df['age'] = np.random.randint(1, 81, size=len(df))
df['weight'] = np.random.uniform(50, 150, size=len(df)).round(1)

# 2. Update 'Target_Column' based on 'New_Column'
# Syntax: np.where(condition, value_if_true, value_if_false)
df['LABEL'] = np.where(df['age'] > 60, df['LABEL']+1,df['LABEL'] )
df['LABEL'] = np.where((df['weight'] >90) | (df['weight']<50) , df['LABEL']+1,df['LABEL'] )

In [ ]:
output_csv_path = '/content/drive/MyDrive/training_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"Modified DataFrame saved to: {output_csv_path}")

# Display the first few rows of the saved DataFrame to confirm
df_saved = pd.read_csv(output_csv_path)
display(df_saved.head())

Modified DataFrame saved to: /content/drive/MyDrive/training_processed.csv


,LABEL,MTPSA+MTPSA,MRVSA9,MRVSA8,MRVSA0,MRVSA2,VSAEstate10+VSAEstate10,EstateVSA0*LabuteASA,PEOEVSA12,PEOEVSA10,...,PEOEVSA9,slogPVSA2,slogPVSA0,slogPVSA9,VSAEstate7+VSAEstate7,EstateVSA7,EstateVSA2,EstateVSA1*VSAEstate8,age,weight
0,1,529.89,73.841,0.0,48.881,42.201,0.000,21369.075,0.0,0.0,...,36.251,52.824,48.685,0.0,5.810,42.201,23.969,4199.470,1,122.7
1,0,525.93,73.841,0.0,48.881,42.201,2.146,20633.988,0.0,0.0,...,36.251,52.824,48.685,0.0,0.000,42.201,23.969,3802.103,3,76.4
2,0,580.67,73.841,0.0,48.881,42.201,0.000,31298.887,0.0,0.0,...,36.251,52.824,48.685,0.0,0.000,42.201,23.969,6002.492,5,83.2
3,1,560.55,73.841,0.0,48.881,42.201,0.000,19925.483,0.0,0.0,...,36.251,52.824,48.685,0.0,10.137,42.201,23.969,4345.541,14,111.7
4,1,1642.76,73.841,0.0,48.881,42.201,0.000,119357.269,0.0,0.0,...,36.251,52.824,48.685,0.0,0.000,42.201,23.969,27894.988,32,96.0


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/01_validation.csv')

# 1. Add the new column (e.g., a simple flag)
df['age'] = np.random.randint(1, 81, size=len(df))
df['weight'] = np.random.uniform(50, 150, size=len(df)).round(1)

# 2. Update 'Target_Column' based on 'New_Column'
# Syntax: np.where(condition, value_if_true, value_if_false)
df['LABEL'] = np.where(df['age'] > 60, df['LABEL']+1,df['LABEL'] )
df['LABEL'] = np.where((df['weight'] >90) | (df['weight']<50) , df['LABEL']+1,df['LABEL'] )

In [ ]:
output_csv_path = '/content/drive/MyDrive/validation_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"Modified DataFrame saved to: {output_csv_path}")

# Display the first few rows of the saved DataFrame to confirm
df_saved = pd.read_csv(output_csv_path)
display(df_saved.head())

Modified DataFrame saved to: /content/drive/MyDrive/validation_processed.csv


,LABEL,MTPSA+MTPSA,MRVSA9,MRVSA8,MRVSA0,MRVSA2,VSAEstate10+VSAEstate10,EstateVSA0*LabuteASA,PEOEVSA12,PEOEVSA10,...,PEOEVSA9,slogPVSA2,slogPVSA0,slogPVSA9,VSAEstate7+VSAEstate7,EstateVSA7,EstateVSA2,EstateVSA1*VSAEstate8,age,weight
0,3,169.64,23.158,28.525,28.537,30.175,0.0,1475.981,6.010,5.824,...,6.104,20.755,4.737,4.795,1.849,25.608,36.020,342.146,65,70.2
1,2,110.13,11.753,0.000,33.320,0.000,0.0,2577.217,0.000,0.000,...,5.783,14.326,0.000,8.781,28.817,0.000,32.104,1115.680,25,119.4
2,1,124.99,5.969,0.000,44.565,0.000,0.0,1259.483,0.000,0.000,...,0.000,4.795,34.664,0.000,0.000,0.000,6.421,3484.858,66,67.6
3,1,293.37,15.192,0.000,33.810,0.000,0.0,7397.889,5.083,0.000,...,0.000,9.130,5.734,0.000,21.434,0.000,0.000,1448.950,18,128.8
4,0,121.54,5.969,0.000,20.114,4.900,0.0,2856.063,0.000,0.000,...,5.601,15.811,0.000,0.000,0.000,4.900,6.421,286.035,59,87.6


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/02_testing.csv')

# 1. Add the new column (e.g., a simple flag)
df['age'] = np.random.randint(1, 81, size=len(df))
df['weight'] = np.random.uniform(50, 150, size=len(df)).round(1)

# 2. Update 'Target_Column' based on 'New_Column'
# Syntax: np.where(condition, value_if_true, value_if_false)
df['LABEL'] = np.where(df['age'] > 60, df['LABEL']+1,df['LABEL'] )
df['LABEL'] = np.where((df['weight'] >90) | (df['weight']<50) , df['LABEL']+1,df['LABEL'] )

In [ ]:
output_csv_path = '/content/drive/MyDrive/testing_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"Modified DataFrame saved to: {output_csv_path}")

# Display the first few rows of the saved DataFrame to confirm
df_saved = pd.read_csv(output_csv_path)
display(df_saved.head())

Modified DataFrame saved to: /content/drive/MyDrive/testing_processed.csv


,LABEL,MTPSA+MTPSA,MRVSA9,MRVSA8,MRVSA0,MRVSA2,VSAEstate10+VSAEstate10,EstateVSA0*LabuteASA,PEOEVSA12,PEOEVSA10,...,PEOEVSA9,slogPVSA2,slogPVSA0,slogPVSA9,VSAEstate7+VSAEstate7,EstateVSA7,EstateVSA2,EstateVSA1*VSAEstate8,age,weight
0,1,111.73,23.382,0.000,0.000,19.936,0.0,0.000,0.0,10.288,...,5.517,0.000,0.000,0.000,0.547,19.936,4.641,0.000,47,148.3
1,3,121.96,17.846,0.000,24.227,10.217,0.0,1030.567,0.0,0.000,...,12.084,25.541,5.317,0.000,0.000,5.317,19.262,476.950,38,100.6
2,1,123.39,11.595,5.750,9.531,20.207,0.0,0.000,0.0,5.824,...,5.750,17.636,10.054,5.687,2.053,25.524,19.070,195.917,40,127.0
3,2,71.44,23.089,0.000,14.696,0.000,0.0,542.023,0.0,0.000,...,0.000,4.795,0.000,0.000,0.000,0.000,15.318,98.120,42,136.2
4,3,195.94,34.009,11.499,18.601,14.868,0.0,1014.880,0.0,11.499,...,17.962,4.737,14.791,15.896,13.468,20.185,35.128,410.630,2,120.0


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/03_external_test_set.csv')

# 1. Add the new column (e.g., a simple flag)
df['age'] = np.random.randint(1, 81, size=len(df))
df['weight'] = np.random.uniform(50, 150, size=len(df)).round(1)

# 2. Update 'Target_Column' based on 'New_Column'
# Syntax: np.where(condition, value_if_true, value_if_false)
df['LABEL'] = np.where(df['age'] > 60, df['LABEL']+1,df['LABEL'] )
df['LABEL'] = np.where((df['weight'] >90) | (df['weight']<50) , df['LABEL']+1,df['LABEL'] )

In [ ]:
output_csv_path = '/content/drive/MyDrive/ext_test_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"Modified DataFrame saved to: {output_csv_path}")

# Display the first few rows of the saved DataFrame to confirm
df_saved = pd.read_csv(output_csv_path)
display(df_saved.head())

Modified DataFrame saved to: /content/drive/MyDrive/ext_test_processed.csv


,MTPSA+MTPSA,MRVSA9,MRVSA8,MRVSA0,MRVSA2,VSAEstate10+VSAEstate10,EstateVSA0*LabuteASA,PEOEVSA12,PEOEVSA10,PEOEVSA5,...,slogPVSA2,slogPVSA0,slogPVSA9,VSAEstate7+VSAEstate7,EstateVSA7,EstateVSA2,EstateVSA1*VSAEstate8,LABEL,age,weight
0,46.26,6.076,12.344,9.630,5.157,0.0,102.173,0.0,5.76,30.498,...,6.421,0.000,0.0,5.432,31.001,17.754,0.000,2,23,121.1
1,109.45,6.076,12.344,9.630,5.157,0.0,125.961,0.0,5.76,30.498,...,6.421,0.000,0.0,5.432,31.001,17.754,245.484,1,4,76.8
2,46.26,6.076,12.344,9.630,5.157,0.0,227.199,0.0,5.76,30.498,...,6.421,0.000,0.0,5.432,31.001,17.754,108.300,2,57,111.5
3,110.33,6.076,12.344,9.630,5.157,0.0,353.754,0.0,5.76,30.498,...,6.421,0.000,0.0,5.432,31.001,17.754,227.430,1,19,67.3
4,418.27,90.521,5.750,48.569,42.201,0.0,2420.131,0.0,0.00,108.232,...,57.619,54.419,0.0,0.000,42.201,23.969,1121.631,1,22,105.3


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score

train_df = pd.read_csv('/content/drive/MyDrive/training_processed.csv')
val_df = pd.read_csv('/content/drive/MyDrive/validation_processed.csv')
test_df = pd.read_csv('/content/drive/MyDrive/testing_processed.csv')
external_df = pd.read_csv('/content/drive/MyDrive/ext_test_processed.csv')

X_train, y_train = train_df.drop('LABEL', axis=1), train_df['LABEL']
X_val, y_val = val_df.drop('LABEL', axis=1), val_df['LABEL']
X_test, y_test = test_df.drop('LABEL', axis=1), test_df['LABEL']
X_ext, y_ext = external_df.drop('LABEL', axis=1), external_df['LABEL']

print("Training the balanced model...")
model = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=0.1,
    reg_alpha=0.0,
    early_stopping_rounds=50
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

try:
    model.save_model("drug_food_model.json")
    print("✅ SUCCESS! 'drug_food_model.json' has been created and saved!")
except FileNotFoundError:
    print("❌ Error: Please make sure '01_validation.csv' is uploaded to Colab first.")

print("\n--- Model Evaluation ---")

test_predictions = model.predict(X_test)
test_acc = accuracy_score(y_test, test_predictions)
print(f"Testing Accuracy (Internal): {test_acc * 100:.2f}%")

ext_predictions = model.predict(X_ext)
ext_acc = accuracy_score(y_ext, ext_predictions)
print(f"External Accuracy (Real-World): {ext_acc * 100:.2f}%")



# Download it to your computer
from google.colab import files
files.download("drug_food_model.json")

Training the balanced model...
✅ SUCCESS! 'drug_food_model.json' has been created and saved!

--- Model Evaluation ---
Testing Accuracy (Internal): 96.49%
External Accuracy (Real-World): 82.21%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import pandas as pd
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.EState.EState_VSA import EState_VSA_, VSA_EState_

class DrugFoodInteractionPredictor:
    def __init__(self, model_path="drug_food_model.json", drugs_csv="drugs.csv", food_csv="food.csv"):
        """Initializes the predictor by loading the model and datasets into memory ONCE."""
        print("[*] Loading datasets and model into memory...")

        try:
            # Load data files once during initialization
            self.drugs_df = pd.read_csv('/content/drive/MyDrive/combined_drug_dataset (1).csv', index_col='name')
            self.food_df = pd.read_csv('/content/drive/MyDrive/food_compound_dataset.csv', index_col='name')

            # Load the XGBoost model once
            self.model = xgb.XGBClassifier()
            self.model.load_model(model_path)
        except FileNotFoundError as e:
            print(f"❌ Initialization Error: Could not find file - {e.filename}")
            raise

        # These column names match your Excel file headers perfectly
        # These column names must perfectly match the exact strings
        # that were generated when the XGBoost model was trained.
        self.feature_columns = [
            'MTPSA+MTPSA',
            'MRVSA9',
            'MRVSA8',
            'MRVSA0',
            'MRVSA2',
            'VSAEstate10+VSAEstate10',
            'EstateVSA0*LabuteASA',
            'PEOEVSA12',
            'PEOEVSA10',
            'PEOEVSA5',
            'PEOEVSA9',
            'slogPVSA2',
            'slogPVSA0',
            'slogPVSA9',
            'VSAEstate7+VSAEstate7',
            'EstateVSA7',
            'EstateVSA2',
            'EstateVSA1*VSAEstate8',
            'age',
            'weight'
        ]
        print("[+] Initialization complete! Ready for predictions.")

    def _get_base_features(self, smiles):
        """Internal helper to calculate RDKit descriptors."""
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            return None

        return {
            'MTPSA': Descriptors.TPSA(mol),
            'LabuteASA': rdMolDescriptors.CalcLabuteASA(mol),
            'MRVSA': rdMolDescriptors.SMR_VSA_(mol),
            'PEOEVSA': rdMolDescriptors.PEOE_VSA_(mol),
            'slogPVSA': rdMolDescriptors.SlogP_VSA_(mol),
            'EstateVSA': EState_VSA_(mol), # Now uses the correctly imported function
            'VSAEstate': VSA_EState_(mol)  # Now uses the correctly imported function
        }

    def predict(self, drug_name, food_name, age, weight):
        """Retrieves features and returns the model prediction."""

        # 1. Retrieve SMILES Strings
        try:
            drug_smiles = self.drugs_df.loc[drug_name, "smiles"]
            food_smiles = self.food_df.loc[food_name, "smiles"]
        except KeyError as e:
            return {"error": f"Item {e} was not found in your CSV database."}

        # 2. Calculate Base Molecular Features
        d = self._get_base_features(drug_smiles)
        f = self._get_base_features(food_smiles)

        if not d or not f:
            return {"error": "Invalid SMILES string. RDKit parsing failed."}

        # 3. Combine Features (Math matches your 18 columns)
        mol_features = [
            d['MTPSA'] + f['MTPSA'],
            d['MRVSA'][9] + f['MRVSA'][9],
            d['MRVSA'][8] + f['MRVSA'][8],
            d['MRVSA'][0] + f['MRVSA'][0],
            d['MRVSA'][2] + f['MRVSA'][2],
            d['VSAEstate'][9] + f['VSAEstate'][9],
            d['EstateVSA'][0] * f['LabuteASA'],
            d['PEOEVSA'][12] + f['PEOEVSA'][12],
            d['PEOEVSA'][10] + f['PEOEVSA'][10],
            d['PEOEVSA'][5] + f['PEOEVSA'][5],
            d['PEOEVSA'][9] + f['PEOEVSA'][9],
            d['slogPVSA'][2] + f['slogPVSA'][2],
            d['slogPVSA'][0] + f['slogPVSA'][0],
            d['slogPVSA'][9] + f['slogPVSA'][9],
            d['VSAEstate'][7] + f['VSAEstate'][7],
            d['EstateVSA'][7] + f['EstateVSA'][7],
            d['EstateVSA'][2] + f['EstateVSA'][2],
            d['EstateVSA'][1] * f['VSAEstate'][8]
        ]

        # 4. Append Age and Weight to make 20 features
        final_features = mol_features + [age, weight]

        # 5. Format and Predict
        input_df = pd.DataFrame([final_features], columns=self.feature_columns)

        prediction = self.model.predict(input_df)

        # Return as a standard Python integer so it easily converts to JSON for your web app
        return {"prediction": int(prediction[0])}


# ==========================================
# HOW TO USE THIS IN YOUR SCRIPT OR API
# ==========================================

if __name__ == "__main__":
    # 1. Initialize the class ONCE at the top of your script/server
    # (Make sure 'drug_food_model.json', 'drugs.csv', and 'food.csv' are in the same folder)
    drive_model_path = "/content/drive/MyDrive/drug_food_model (1).json"
    predictor = DrugFoodInteractionPredictor(model_path=drive_model_path)

    # 2. Call the .predict() method whenever a user submits a form
    result = predictor.predict(
        drug_name="Aspirin",
        food_name="Dioscin",
        age=81,
        weight=150
    )

    print(f"Prediction Result: {result}")

[*] Loading datasets and model into memory...
[+] Initialization complete! Ready for predictions.
Prediction Result: {'prediction': 2}
